# M2.S5 - Asynchronous Guided Lab
## One application, three parallel models, one real HPC system

**Estimated time: 90 minutes**

In the four live sessions of Module 2 you learned how to think about parallelism, how OpenMP uses shared memory, how MPI uses separate processes and explicit communication, and how GPUs accelerate highly data-parallel work.

This guided lab brings those ideas together around **one small but realistic scientific application**: two-dimensional heat diffusion.

You will run the same numerical problem in several ways:

```text
serial CPU
   |
   +--> OpenMP on several CPU cores
   |
   +--> MPI with several independent processes
   |
   +--> optional GPU extension with OpenACC
```

The goal is not to write a large application. The goal is to understand what changes when the **programming model** changes.

### Suggested timing

| Part | Time |
|---|---:|
| Understand the application | 10 min |
| Serial baseline and correctness | 10 min |
| OpenMP experiment | 20 min |
| MPI experiment | 25 min |
| Compare and explain | 10 min |
| GPU extension | 15 min |

> **Core assessed path:** serial + OpenMP + MPI.  
> **GPU extension:** strongly recommended, but separated so the core lab remains aligned with Practice 2 in the syllabus.

### Before you start

In JupyterLab use **Kernel -> Restart Kernel and Run All Cells**.

Do not jump directly to the Slurm submission cells. Earlier cells create the source files and job scripts used later in the lab.

Throughout the notebook use the same cycle:

> **PREDICT -> RUN -> OBSERVE -> EXPLAIN**

> **Notebook build: M2S5-2026-09-20-v2**

## 1 - The application: heat spreading across a plate

Imagine a square metal plate. At the beginning, one point in the center is hot and the rest is cold.

At each simulation step, every interior grid point becomes the average of its four direct neighbours:

```text
             north
               |
west ---- current point ---- east
               |
             south

new(i,j) = 0.25 * (north + south + west + east)
```

This is a **5-point stencil**.

Stencil computations appear in real scientific codes for:

- heat transfer and diffusion;
- computational fluid dynamics;
- weather and climate models;
- image processing;
- numerical solutions of partial differential equations.

The mathematical problem is deliberately simple so that we can focus on **parallel execution**.

### Predict before running

Think about the update of two different grid cells during the **same timestep**.

1. Can they usually be computed independently?
2. Why do we still need synchronization between timesteps?
3. Which programming model will require explicit communication when the grid is split into pieces?

<details>
<summary><strong>Show explanation</strong></summary>

Within one timestep, different output cells can be computed in parallel because they read the old grid and write different locations in the new grid.

Before the next timestep starts, all updates for the current timestep must be complete. That creates a synchronization point.

With OpenMP, threads share the same arrays in one memory space. With MPI, each process owns only part of the grid, so neighbouring processes must explicitly exchange boundary or **halo** rows.

</details>

In [ ]:
from pathlib import Path
import os
import re
import subprocess
import time
import math

import numpy as np
from IPython.display import display, HTML, SVG

GRID_N = 2048
STEPS = 200
OMP_THREADS = [1, 2, 4, 8]
MPI_RANKS = [1, 2, 4]

def require_files(*paths):
    missing = [str(p) for p in paths if not Path(p).exists()]
    if missing:
        raise RuntimeError(
            "Missing file(s): " + ", ".join(missing) +
            ". Run the notebook from the top with Kernel -> Restart Kernel and Run All Cells."
        )

def _heat_color(value, vmax=100.0):
    # Square-root scaling makes the spreading heat visible while preserving
    # a common scale across snapshots.
    t = 0.0 if vmax <= 0 else max(0.0, min(1.0, math.sqrt(value / vmax)))
    if t < 0.5:
        r = int(510 * t)
        g = 0
    else:
        r = 255
        g = int(510 * (t - 0.5))
    b = 0
    return f"rgb({r},{g},{b})"

def show_heat_snapshots(snapshots, labels):
    panels = []
    for arr, label in zip(snapshots, labels):
        small = arr[::2, ::2]
        rects = []
        cell = 5
        for i in range(small.shape[0]):
            for j in range(small.shape[1]):
                rects.append(
                    f'<rect x="{j*cell}" y="{i*cell}" width="{cell}" height="{cell}" '
                    f'fill="{_heat_color(float(small[i,j]))}" />'
                )
        w = small.shape[1] * cell
        h = small.shape[0] * cell
        panels.append(
            f'<div style="text-align:center;margin-right:14px">'
            f'<div style="font-weight:600;margin-bottom:4px">{label}</div>'
            f'<svg width="{w}" height="{h}" viewBox="0 0 {w} {h}">'
            + "".join(rects) + "</svg></div>"
        )
    display(HTML('<div style="display:flex;align-items:flex-start;flex-wrap:wrap">' + "".join(panels) + "</div>"))

def show_speedup_chart(workers, speedups, title):
    workers = [float(x) for x in workers]
    speedups = [float(x) for x in speedups]
    width, height = 560, 330
    left, right, top, bottom = 60, 25, 35, 55
    pw = width - left - right
    ph = height - top - bottom
    max_x = max(workers) if workers else 1.0
    max_y = max(max(speedups, default=1.0), max_x) * 1.10

    def X(x):
        return left + (x / max_x) * pw

    def Y(y):
        return top + ph - (y / max_y) * ph

    parts = [
        f'<svg width="{width}" height="{height}" viewBox="0 0 {width} {height}" xmlns="http://www.w3.org/2000/svg">',
        f'<text x="{width/2}" y="20" text-anchor="middle" font-size="16" font-weight="600">{title}</text>',
        f'<line x1="{left}" y1="{top}" x2="{left}" y2="{top+ph}" stroke="#444"/>',
        f'<line x1="{left}" y1="{top+ph}" x2="{left+pw}" y2="{top+ph}" stroke="#444"/>',
        f'<line x1="{X(0)}" y1="{Y(0)}" x2="{X(max_x)}" y2="{Y(max_x)}" stroke="#888" stroke-dasharray="5,5"/>'
    ]

    points = " ".join(f"{X(x)},{Y(y)}" for x, y in zip(workers, speedups))
    parts.append(f'<polyline points="{points}" fill="none" stroke="#2463a6" stroke-width="2.5"/>')
    for x, y in zip(workers, speedups):
        parts.append(f'<circle cx="{X(x)}" cy="{Y(y)}" r="4" fill="#2463a6"/>')
        parts.append(f'<text x="{X(x)}" y="{top+ph+20}" text-anchor="middle" font-size="12">{int(x)}</text>')
        parts.append(f'<text x="{X(x)+7}" y="{Y(y)-7}" font-size="11">{y:.2f}x</text>')

    parts.append(f'<text x="{width/2}" y="{height-10}" text-anchor="middle" font-size="13">workers</text>')
    parts.append(f'<text x="16" y="{top+ph/2}" text-anchor="middle" font-size="13" transform="rotate(-90 16 {top+ph/2})">speedup vs serial</text>')
    parts.append('</svg>')
    display(SVG("".join(parts)))

print("M2.S5 guided lab")
print("Large benchmark grid:", GRID_N, "x", GRID_N)
print("Timesteps:", STEPS)
print("OpenMP thread counts:", OMP_THREADS)
print("MPI rank counts:", MPI_RANKS)
print("Notebook environment: PASS")

### Make the application visible

The next cell runs a tiny Python version only for visualization.

This is **not** the performance implementation. The real benchmark later is compiled C code submitted through Slurm.

In [ ]:
def heat_demo(n=41, steps=30):
    u = np.zeros((n, n), dtype=float)
    u[n//2, n//2] = 100.0

    snapshots = [u.copy()]
    for step in range(steps):
        v = np.zeros_like(u)
        v[1:-1, 1:-1] = 0.25 * (
            u[:-2, 1:-1] + u[2:, 1:-1] +
            u[1:-1, :-2] + u[1:-1, 2:]
        )
        u = v
        if step in {3, 9, steps-1}:
            snapshots.append(u.copy())
    return snapshots

snaps = heat_demo()
show_heat_snapshots(snaps, ["start", "4 steps", "10 steps", "30 steps"])

## 2 - Tools for the real cluster runs

The remaining performance experiments use the SciTech Slurm cluster.

The helper functions below:

- remove inherited Jupyter Slurm memory variables before a new `sbatch`;
- submit a job;
- show its queue state;
- wait for completion;
- print the corresponding output file.

You do not need to modify these helpers.

In [ ]:
def _clean_submit_env():
    env = os.environ.copy()
    for key in ("SLURM_MEM_PER_CPU", "SLURM_MEM_PER_GPU", "SLURM_MEM_PER_NODE"):
        env.pop(key, None)
    return env

def submit_slurm(script_path):
    p = subprocess.run(
        ["sbatch", "--parsable", str(script_path)],
        capture_output=True, text=True, env=_clean_submit_env()
    )
    if p.returncode != 0:
        raise RuntimeError(p.stderr.strip() or p.stdout.strip())
    job_id = p.stdout.strip().split(";")[0]
    print("Submitted Slurm job:", job_id)
    return job_id

def slurm_status(job_id):
    if not job_id:
        print("No job id.")
        return
    p = subprocess.run(
        ["squeue", "-j", str(job_id), "-o", "%.18i %.10T %.20R %.12M"],
        capture_output=True, text=True
    )
    print(p.stdout.strip() or f"Job {job_id} has left the queue.")

def wait_for_job(job_id, timeout=480, poll=4):
    start = time.time()
    while time.time() - start < timeout:
        p = subprocess.run(
            ["squeue", "-h", "-j", str(job_id), "-o", "%T"],
            capture_output=True, text=True
        )
        state = p.stdout.strip()
        if not state:
            time.sleep(2)
            print(f"Job {job_id} has completed or left the queue.")
            return True
        print(f"Job {job_id}: {state}")
        time.sleep(poll)
    print("Timed out waiting for the job.")
    return False

def show_job_output(job_id, prefix):
    path = Path(f"{prefix}-{job_id}.out")
    if not path.exists():
        print("Output file not found yet:", path)
        return ""
    out = path.read_text(errors="replace")
    print(out)
    return out

print("Slurm helpers ready.")

## 3 - Serial baseline

We first need a correct baseline.

For a tiny deterministic test:

- grid = 21 x 21;
- timesteps = 4;
- initial center temperature = 100.

A correct implementation should produce approximately:

```text
CENTER_VALUE=14.062500
CHECKSUM=100.000000
```

The checksum is useful because every parallel version should solve the **same problem**, not merely run fast.

In [ ]:
serial_src = r'''
#include <stdio.h>
#include <stdlib.h>
#include <string.h>
#include <time.h>
#include <math.h>

static double now_s(void){
    struct timespec ts;
    clock_gettime(CLOCK_MONOTONIC, &ts);
    return ts.tv_sec + ts.tv_nsec/1e9;
}

static void parse(int argc, char **argv, int *n, int *steps){
    *n = 2048;
    *steps = 200;
    for(int i=1; i<argc; ++i){
        if(!strcmp(argv[i],"--demo")){ *n=21; *steps=4; }
        else if(!strcmp(argv[i],"--n") && i+1<argc) *n=atoi(argv[++i]);
        else if(!strcmp(argv[i],"--steps") && i+1<argc) *steps=atoi(argv[++i]);
    }
}

int main(int argc, char **argv){
    int n, steps;
    parse(argc, argv, &n, &steps);

    size_t sz = (size_t)n*n;
    double *u = calloc(sz, sizeof(double));
    double *v = calloc(sz, sizeof(double));
    if(!u || !v) return 2;

    u[(size_t)(n/2)*n + n/2] = 100.0;

    double t0 = now_s();
    for(int s=0; s<steps; ++s){
        memset(v, 0, sz*sizeof(double));
        for(int i=1; i<n-1; ++i)
            for(int j=1; j<n-1; ++j)
                v[(size_t)i*n+j] = 0.25 * (
                    u[(size_t)(i-1)*n+j] +
                    u[(size_t)(i+1)*n+j] +
                    u[(size_t)i*n+j-1] +
                    u[(size_t)i*n+j+1]
                );
        double *tmp=u; u=v; v=tmp;
    }
    double t1 = now_s();

    double sum=0.0;
    for(size_t k=0; k<sz; ++k) sum += u[k];

    printf("CENTER_VALUE=%.6f\n", u[(size_t)(n/2)*n+n/2]);
    printf("CHECKSUM=%.6f\n", sum);
    printf("RESULT model=serial workers=1 seconds=%.6f checksum=%.6f\n",
           t1-t0, sum);

    free(u); free(v);
    return 0;
}
'''

Path("m2s5_serial.c").write_text(serial_src)
print("Wrote m2s5_serial.c")

In [ ]:
p = subprocess.run(
    ["bash", "-lc", "gcc -O3 -std=gnu11 m2s5_serial.c -lm -o m2s5_serial && ./m2s5_serial --demo"],
    capture_output=True, text=True
)
print(p.stdout)
if p.stderr:
    print(p.stderr)

assert "CENTER_VALUE=14.062500" in p.stdout
assert "CHECKSUM=100.000000" in p.stdout
print("Correctness check: PASS")

### Checkpoint

Why do we establish correctness **before** measuring speed?

<details>
<summary><strong>Show explanation</strong></summary>

A faster program is useless if it computes a different result. Parallelization can introduce races, missing communication, incorrect synchronization or decomposition errors. A deterministic reference result gives us a simple correctness test before interpreting performance numbers.

</details>

## 4 - OpenMP: shared-memory parallelism

OpenMP keeps one process and creates several threads inside it.

All threads can access the same `current` and `next` arrays.

The key parallel loop is:

```c
#pragma omp parallel for collapse(2) schedule(static)
for (int i=1; i<n-1; ++i)
    for (int j=1; j<n-1; ++j)
        ...
```

### Predict

Before running:

1. Do OpenMP threads need to send halo rows to each other?
2. Will 8 threads necessarily be exactly 8 times faster than one thread?
3. What shared hardware resource may eventually limit scaling?

<details>
<summary><strong>Show explanation</strong></summary>

Threads share memory, so no explicit halo messages are needed.

Speedup is rarely perfectly linear because of parallel overhead, synchronization and shared hardware limits. For a stencil, memory bandwidth can become an important bottleneck as more cores access the arrays concurrently.

</details>

In [ ]:
openmp_src = r'''
#include <stdio.h>
#include <stdlib.h>
#include <string.h>
#include <math.h>
#include <omp.h>

static void parse(int argc, char **argv, int *n, int *steps){
    *n = 2048;
    *steps = 200;
    for(int i=1; i<argc; ++i){
        if(!strcmp(argv[i],"--demo")){ *n=21; *steps=4; }
        else if(!strcmp(argv[i],"--n") && i+1<argc) *n=atoi(argv[++i]);
        else if(!strcmp(argv[i],"--steps") && i+1<argc) *steps=atoi(argv[++i]);
    }
}

int main(int argc, char **argv){
    int n, steps;
    parse(argc, argv, &n, &steps);

    size_t sz = (size_t)n*n;
    double *u = calloc(sz, sizeof(double));
    double *v = calloc(sz, sizeof(double));
    if(!u || !v) return 2;

    u[(size_t)(n/2)*n + n/2] = 100.0;

    double t0 = omp_get_wtime();
    for(int s=0; s<steps; ++s){
        memset(v, 0, sz*sizeof(double));

        #pragma omp parallel for collapse(2) schedule(static)
        for(int i=1; i<n-1; ++i)
            for(int j=1; j<n-1; ++j)
                v[(size_t)i*n+j] = 0.25 * (
                    u[(size_t)(i-1)*n+j] +
                    u[(size_t)(i+1)*n+j] +
                    u[(size_t)i*n+j-1] +
                    u[(size_t)i*n+j+1]
                );

        double *tmp=u; u=v; v=tmp;
    }
    double t1 = omp_get_wtime();

    double sum=0.0;
    for(size_t k=0; k<sz; ++k) sum += u[k];

    printf("CENTER_VALUE=%.6f\n", u[(size_t)(n/2)*n+n/2]);
    printf("RESULT model=openmp workers=%d seconds=%.6f checksum=%.6f\n",
           omp_get_max_threads(), t1-t0, sum);

    free(u); free(v);
    return 0;
}
'''

Path("m2s5_openmp.c").write_text(openmp_src)
print("Wrote m2s5_openmp.c")

### Run the serial and OpenMP versions on a real CPU allocation

One Slurm job requests 8 CPU cores and runs:

- serial baseline;
- OpenMP with 1 thread;
- OpenMP with 2 threads;
- OpenMP with 4 threads;
- OpenMP with 8 threads.

Using one allocation keeps the hardware context consistent.

In [ ]:
cpu_script = f'''#!/bin/bash
#SBATCH --job-name=m2s5_cpu
#SBATCH --partition=cpu
#SBATCH --nodes=1
#SBATCH --ntasks=1
#SBATCH --cpus-per-task=8
#SBATCH --mem=4G
#SBATCH --time=00:06:00
#SBATCH --output=m2s5_cpu-%j.out

set -e
cd "$SLURM_SUBMIT_DIR"

unset SLURM_MEM_PER_CPU
unset SLURM_MEM_PER_GPU
unset SLURM_MEM_PER_NODE

echo "=== M2.S5 CPU / OPENMP ==="
echo "HOST=$(hostname)"
echo "CPUS_PER_TASK=$SLURM_CPUS_PER_TASK"

gcc -O3 -std=gnu11 -march=native m2s5_serial.c -lm -o m2s5_serial
gcc -O3 -std=gnu11 -march=native -fopenmp m2s5_openmp.c -lm -o m2s5_openmp

echo
echo "--- serial ---"
./m2s5_serial --n {GRID_N} --steps {STEPS}

for t in 1 2 4 8; do
    echo
    echo "--- openmp $t threads ---"
    OMP_NUM_THREADS=$t OMP_PROC_BIND=close OMP_PLACES=cores \
        ./m2s5_openmp --n {GRID_N} --steps {STEPS}
done
'''

Path("m2s5_cpu.slurm").write_text(cpu_script)
print(cpu_script)

In [ ]:
require_files("m2s5_cpu.slurm", "m2s5_serial.c", "m2s5_openmp.c")
M2S5_CPU_JOB = submit_slurm("m2s5_cpu.slurm")
slurm_status(M2S5_CPU_JOB)

In [ ]:
wait_for_job(M2S5_CPU_JOB)
CPU_OUTPUT = show_job_output(M2S5_CPU_JOB, "m2s5_cpu")

### Observe the CPU results

Look for lines beginning with `RESULT`.

Record:

- serial time;
- OpenMP 1-thread time;
- OpenMP 2-thread time;
- OpenMP 4-thread time;
- OpenMP 8-thread time.

Do all versions report approximately the same checksum?

If not, stop and investigate correctness before comparing speed.

In [ ]:
result_re = re.compile(
    r"RESULT model=(\w+) workers=(\S+) seconds=([0-9.eE+-]+) checksum=([0-9.eE+-]+)"
)

def parse_results(text):
    rows = []
    for m in result_re.finditer(text or ""):
        model, workers, seconds, checksum = m.groups()
        rows.append({
            "model": model,
            "workers": workers,
            "seconds": float(seconds),
            "checksum": float(checksum),
        })
    return rows

cpu_rows = parse_results(CPU_OUTPUT)

print(f"{'model':<10} {'workers':>8} {'seconds':>12} {'speedup':>10}")
serial_time = next(r["seconds"] for r in cpu_rows if r["model"]=="serial")
for r in cpu_rows:
    speedup = serial_time / r["seconds"]
    print(f'{r["model"]:<10} {r["workers"]:>8} {r["seconds"]:12.6f} {speedup:10.2f}')

omp_rows = [r for r in cpu_rows if r["model"]=="openmp"]
show_speedup_chart(
    [int(r["workers"]) for r in omp_rows],
    [serial_time/r["seconds"] for r in omp_rows],
    "OpenMP scaling"
)

### Explain

Complete these sentences in your own words before revealing the explanation.

- OpenMP uses __________ memory.
- A thread is different from an MPI rank because __________.
- My measured scaling becomes less than ideal because __________.

<details>
<summary><strong>Show suggested explanation</strong></summary>

OpenMP uses **shared memory** inside one process. Threads can access the same arrays directly.

An MPI rank is an independent process with its own address space. An OpenMP thread shares its process memory with the other threads.

Scaling becomes less than ideal because parallel execution has overhead and because resources such as memory bandwidth are shared. The stencil performs many memory accesses relative to arithmetic work.

</details>

## 5 - MPI: distributed-memory thinking

Now we solve the same grid with independent MPI processes.

Each rank owns a horizontal strip:

```text
rank 0: rows 0 ...........
rank 1: rows .............
rank 2: rows .............
rank 3: rows .......... N-1
```

To update the first and last owned rows, a rank needs data from its neighbours.

We therefore add **halo rows**:

```text
upper halo       <- received from rank above
owned rows
lower halo       <- received from rank below
```

### Predict

For four ranks, which ranks have two real neighbours?

What must happen before each timestep can compute its boundary cells?

<details>
<summary><strong>Show explanation</strong></summary>

Middle ranks have an upper and lower neighbour. The first and last ranks have one real neighbour and one physical domain boundary.

Before computing the next stencil step, neighbouring ranks exchange their boundary rows so each process has up-to-date halo data.

</details>

In [ ]:
mpi_src = r'''
#include <mpi.h>
#include <stdio.h>
#include <stdlib.h>
#include <string.h>
#include <math.h>

static void parse(int argc, char **argv, int *n, int *steps){
    *n = 2048;
    *steps = 200;
    for(int i=1; i<argc; ++i){
        if(!strcmp(argv[i],"--demo")){ *n=21; *steps=4; }
        else if(!strcmp(argv[i],"--n") && i+1<argc) *n=atoi(argv[++i]);
        else if(!strcmp(argv[i],"--steps") && i+1<argc) *steps=atoi(argv[++i]);
    }
}

static void exchange_halos(double *u, int myrows, int n, int rank, int size){
    int up = (rank == 0) ? MPI_PROC_NULL : rank-1;
    int down = (rank == size-1) ? MPI_PROC_NULL : rank+1;

    MPI_Sendrecv(
        &u[(size_t)1*n], n, MPI_DOUBLE, up, 10,
        &u[(size_t)(myrows+1)*n], n, MPI_DOUBLE, down, 10,
        MPI_COMM_WORLD, MPI_STATUS_IGNORE
    );

    MPI_Sendrecv(
        &u[(size_t)myrows*n], n, MPI_DOUBLE, down, 20,
        &u[0], n, MPI_DOUBLE, up, 20,
        MPI_COMM_WORLD, MPI_STATUS_IGNORE
    );
}

int main(int argc, char **argv){
    MPI_Init(&argc, &argv);

    int rank, size;
    MPI_Comm_rank(MPI_COMM_WORLD, &rank);
    MPI_Comm_size(MPI_COMM_WORLD, &size);

    char host[MPI_MAX_PROCESSOR_NAME];
    int hlen=0;
    MPI_Get_processor_name(host, &hlen);

    int n, steps;
    parse(argc, argv, &n, &steps);

    int base = n / size;
    int extra = n % size;
    int myrows = base + (rank < extra ? 1 : 0);
    int start = rank*base + (rank < extra ? rank : extra);
    int end = start + myrows - 1;

    double *u = calloc((size_t)(myrows+2)*n, sizeof(double));
    double *v = calloc((size_t)(myrows+2)*n, sizeof(double));
    if(!u || !v) MPI_Abort(MPI_COMM_WORLD, 2);

    int center = n/2;
    if(center >= start && center <= end)
        u[(size_t)(center-start+1)*n + center] = 100.0;

    printf("RANK=%d HOST=%s ROWS=%d-%d\n", rank, host, start, end);
    fflush(stdout);

    MPI_Barrier(MPI_COMM_WORLD);
    double t0 = MPI_Wtime();

    for(int s=0; s<steps; ++s){
        exchange_halos(u, myrows, n, rank, size);
        memset(v, 0, (size_t)(myrows+2)*n*sizeof(double));

        for(int li=1; li<=myrows; ++li){
            int gi = start + li - 1;
            if(gi==0 || gi==n-1) continue;

            for(int j=1; j<n-1; ++j)
                v[(size_t)li*n+j] = 0.25 * (
                    u[(size_t)(li-1)*n+j] +
                    u[(size_t)(li+1)*n+j] +
                    u[(size_t)li*n+j-1] +
                    u[(size_t)li*n+j+1]
                );
        }

        double *tmp=u; u=v; v=tmp;
    }

    MPI_Barrier(MPI_COMM_WORLD);
    double local_time = MPI_Wtime() - t0;

    double local_sum=0.0;
    for(int li=1; li<=myrows; ++li)
        for(int j=0; j<n; ++j)
            local_sum += u[(size_t)li*n+j];

    double global_sum=0.0, max_time=0.0;
    MPI_Reduce(&local_sum, &global_sum, 1, MPI_DOUBLE, MPI_SUM, 0, MPI_COMM_WORLD);
    MPI_Reduce(&local_time, &max_time, 1, MPI_DOUBLE, MPI_MAX, 0, MPI_COMM_WORLD);

    if(rank==0)
        printf("RESULT model=mpi workers=%d seconds=%.6f checksum=%.6f\n",
               size, max_time, global_sum);

    free(u); free(v);
    MPI_Finalize();
    return 0;
}
'''

Path("m2s5_mpi.c").write_text(mpi_src)
print("Wrote m2s5_mpi.c")

### Real MPI job

The current SciTech teaching environment reliably supports real MPI processes on one allocated CPU node.

That is sufficient to demonstrate the MPI programming model correctly:

- separate processes;
- separate address spaces;
- rank IDs;
- explicit halo exchange;
- collective reduction.

A true multi-node MPI software environment is being treated separately as a cluster configuration issue, so this guided student lab does **not** depend on it.

In [ ]:
mpi_script = f'''#!/bin/bash
#SBATCH --job-name=m2s5_mpi
#SBATCH --partition=cpu
#SBATCH --nodes=1
#SBATCH --ntasks=4
#SBATCH --cpus-per-task=1
#SBATCH --mem=4G
#SBATCH --time=00:07:00
#SBATCH --output=m2s5_mpi-%j.out

set -e
cd "$SLURM_SUBMIT_DIR"

unset SLURM_MEM_PER_CPU
unset SLURM_MEM_PER_GPU
unset SLURM_MEM_PER_NODE

echo "=== M2.S5 MPI ==="
echo "HOST=$(hostname)"
echo "ALLOCATED_TASKS=$SLURM_NTASKS"
echo "MPICC=$(command -v mpicc)"
echo "MPIRUN=$(command -v mpirun)"

mpicc -O3 -std=gnu11 -march=native m2s5_mpi.c -lm -o m2s5_mpi

for r in 1 2 4; do
    echo
    echo "--- mpi $r ranks ---"
    mpirun --mca plm ^slurm \
           -np $r \
           --bind-to core \
           --mca pml ob1 \
           --mca btl self,vader,tcp \
           ./m2s5_mpi --n {GRID_N} --steps {STEPS}
done
'''

Path("m2s5_mpi.slurm").write_text(mpi_script)
print(mpi_script)

In [ ]:
require_files("m2s5_mpi.slurm", "m2s5_mpi.c")
M2S5_MPI_JOB = submit_slurm("m2s5_mpi.slurm")
slurm_status(M2S5_MPI_JOB)

In [ ]:
wait_for_job(M2S5_MPI_JOB)
MPI_OUTPUT = show_job_output(M2S5_MPI_JOB, "m2s5_mpi")

### Observe the MPI result

Look for two different kinds of evidence.

**Placement evidence**

```text
RANK=0 HOST=... ROWS=...
RANK=1 HOST=... ROWS=...
...
```

This tells you which rows belong to each rank.

**Performance evidence**

```text
RESULT model=mpi workers=...
```

### Explain

Why does MPI need halo exchange while OpenMP did not?

<details>
<summary><strong>Show explanation</strong></summary>

OpenMP threads can directly read neighbouring rows because all threads share the same arrays.

MPI ranks own separate memory. A rank cannot directly read another rank's rows. It must explicitly send its boundary data and receive neighbouring boundary data into halo rows.

That explicit data movement is the fundamental difference between shared-memory and distributed-memory programming.

</details>

In [ ]:
mpi_rows = parse_results(MPI_OUTPUT)

print(f"{'model':<10} {'ranks':>8} {'seconds':>12} {'speedup vs serial':>18}")
for r in mpi_rows:
    speedup = serial_time / r["seconds"]
    print(f'{r["model"]:<10} {r["workers"]:>8} {r["seconds"]:12.6f} {speedup:18.2f}')

show_speedup_chart(
    [int(r["workers"]) for r in mpi_rows],
    [serial_time/r["seconds"] for r in mpi_rows],
    "MPI scaling on one allocated node"
)

## 6 - Compare the programming models

You have now solved the same scientific problem using:

| Model | Parallel workers | Memory model | How neighbours get data |
|---|---|---|---|
| Serial | 1 process | local | direct memory access |
| OpenMP | threads | shared | direct shared-memory access |
| MPI | processes/ranks | distributed | explicit messages |

### Important interpretation

Do **not** reduce the lesson to "whichever timing is lowest is the best model."

The models solve different scaling problems.

- OpenMP is natural inside one shared-memory node.
- MPI is designed to scale across independent processes and, in production systems, across nodes.
- Performance depends on workload size, communication, synchronization and memory behavior.

The current MPI experiment intentionally stays on one node because that is the validated SciTech student environment today.

In [ ]:
all_core = cpu_rows + mpi_rows

print(f"{'model':<10} {'workers':>8} {'seconds':>12} {'checksum':>14}")
for r in all_core:
    print(f'{r["model"]:<10} {r["workers"]:>8} {r["seconds"]:12.6f} {r["checksum"]:14.6f}')

checksums = [r["checksum"] for r in all_core]
if checksums:
    spread = max(checksums) - min(checksums)
    print("\nChecksum spread:", spread)
    print("Correctness across models:", "PASS" if spread < 1e-6 else "CHECK RESULTS")

## 7 - GPU extension: same stencil with OpenACC

This final part connects the asynchronous practice to M2.S4.

OpenACC lets us keep the numerical loop recognizable while asking the compiler to execute it on the GPU.

Two ideas matter:

1. **parallel work** - many grid cells can be updated simultaneously;
2. **data residence** - the grid should remain on the GPU across many timesteps instead of being transferred every step.

### Predict

For this application, why would moving the two large grids to and from the GPU at every timestep be a bad design?

<details>
<summary><strong>Show explanation</strong></summary>

The stencil performs a relatively small amount of arithmetic per grid point. Repeated host-device transfers could cost more time than the computation itself. Keeping the arrays resident on the GPU allows many timesteps to reuse the same device-resident data.

</details>

In [ ]:
openacc_src = r'''
#include <stdio.h>
#include <stdlib.h>
#include <string.h>
#include <time.h>

static double now_s(void){
    struct timespec ts;
    clock_gettime(CLOCK_MONOTONIC, &ts);
    return ts.tv_sec + ts.tv_nsec/1e9;
}

static void parse(int argc, char **argv, int *n, int *steps){
    *n = 2048;
    *steps = 200;
    for(int i=1; i<argc; ++i){
        if(!strcmp(argv[i],"--demo")){ *n=21; *steps=4; }
        else if(!strcmp(argv[i],"--n") && i+1<argc) *n=atoi(argv[++i]);
        else if(!strcmp(argv[i],"--steps") && i+1<argc) *steps=atoi(argv[++i]);
    }
}

int main(int argc, char **argv){
    int n, steps;
    parse(argc, argv, &n, &steps);

    size_t sz = (size_t)n*n;
    double *current = calloc(sz, sizeof(double));
    double *next = calloc(sz, sizeof(double));
    if(!current || !next) return 2;

    current[(size_t)(n/2)*n + n/2] = 100.0;

    double t0 = now_s();

    #pragma acc data copy(current[0:sz], next[0:sz])
    {
        for(int s=0; s<steps; ++s){
            #pragma acc parallel loop collapse(2) present(current[0:sz], next[0:sz])
            for(int i=1; i<n-1; ++i)
                for(int j=1; j<n-1; ++j)
                    next[(size_t)i*n+j] = 0.25 * (
                        current[(size_t)(i-1)*n+j] +
                        current[(size_t)(i+1)*n+j] +
                        current[(size_t)i*n+j-1] +
                        current[(size_t)i*n+j+1]
                    );

            double *tmp=current; current=next; next=tmp;
        }
    }

    double t1 = now_s();

    double sum=0.0;
    for(size_t k=0; k<sz; ++k) sum += current[k];

    printf("RESULT model=openacc workers=GPU seconds=%.6f checksum=%.6f\n",
           t1-t0, sum);

    free(current); free(next);
    return 0;
}
'''

Path("m2s5_openacc.c").write_text(openacc_src)
print("Wrote m2s5_openacc.c")

### Run on the real SciTech GPU

The validated classroom route is:

```text
Slurm gpu partition
    -> NVIDIA RTX 6000 Ada
    -> nvhpc/25.7
    -> nvc OpenACC compiler
```

Apptainer is not required for the normal path. It remains a fallback option only.

In [ ]:
gpu_script = f'''#!/bin/bash
#SBATCH --job-name=m2s5_gpu
#SBATCH --partition=gpu
#SBATCH --gpus=1
#SBATCH --cpus-per-task=2
#SBATCH --mem=4G
#SBATCH --time=00:07:00
#SBATCH --output=m2s5_gpu-%j.out

set -e
cd "$SLURM_SUBMIT_DIR"

unset SLURM_MEM_PER_CPU
unset SLURM_MEM_PER_GPU
unset SLURM_MEM_PER_NODE

echo "=== M2.S5 OPENACC GPU ==="
echo "HOST=$(hostname)"
nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv,noheader

bash -lc '
    set -e
    module load nvhpc/25.7
    cd "$SLURM_SUBMIT_DIR"
    echo "NVC=$(command -v nvc)"
    nvc --version | head -n 3
    nvc -O3 -acc -Minfo=accel m2s5_openacc.c -o m2s5_openacc
    ./m2s5_openacc --n {GRID_N} --steps {STEPS}
'
'''

Path("m2s5_gpu.slurm").write_text(gpu_script)
print(gpu_script)

In [ ]:
require_files("m2s5_gpu.slurm", "m2s5_openacc.c")
M2S5_GPU_JOB = submit_slurm("m2s5_gpu.slurm")
slurm_status(M2S5_GPU_JOB)

In [ ]:
wait_for_job(M2S5_GPU_JOB)
GPU_OUTPUT = show_job_output(M2S5_GPU_JOB, "m2s5_gpu")

In [ ]:
gpu_rows = parse_results(GPU_OUTPUT)
all_rows = cpu_rows + mpi_rows + gpu_rows

print(f"{'model':<10} {'workers':>8} {'seconds':>12} {'speedup vs serial':>18} {'checksum':>14}")
for r in all_rows:
    speedup = serial_time / r["seconds"]
    print(f'{r["model"]:<10} {r["workers"]:>8} {r["seconds"]:12.6f} {speedup:18.2f} {r["checksum"]:14.6f}')

if gpu_rows:
    print("\nGPU extension: PASS")
else:
    print("\nGPU extension: no GPU result found - inspect the job output above.")

## 8 - Final interpretation

You have now seen one numerical kernel mapped to three different parallel execution models.

### Complete these five reflections

Write short answers in the markdown cell below.

1. **Parallel structure:** Why is the stencil parallel within one timestep but synchronized between timesteps?
2. **OpenMP:** What makes OpenMP convenient for this application inside one node?
3. **MPI:** What new responsibility appears when the grid is split between ranks?
4. **Performance:** Did adding workers always give proportional speedup? What limited scaling?
5. **GPU:** Why is keeping data resident on the GPU important?

### Final decision question

Imagine the grid becomes so large that it no longer fits in one node's memory.

Which model becomes essential, and what additional cost appears?

<details>
<summary><strong>Show suggested answer</strong></summary>

MPI becomes essential because the global data must be distributed across multiple address spaces and potentially multiple nodes. The additional cost is communication, especially the repeated exchange of halo data between neighbouring subdomains. A production application might then combine MPI between nodes, OpenMP within each node, and GPUs for local stencil kernels.

</details>

### Student reflection - edit this cell

**1. Parallel structure:**  
Write your answer here.

**2. OpenMP:**  
Write your answer here.

**3. MPI:**  
Write your answer here.

**4. Performance:**  
Write your answer here.

**5. GPU/data movement:**  
Write your answer here.

**Final decision:**  
Write your answer here.

## 9 - Suggested deliverable

This notebook can itself become the Practice 2 technical log.

Before submitting:

- make sure the serial correctness test says **PASS**;
- keep the CPU/OpenMP Slurm output;
- keep the MPI rank/decomposition output;
- keep the timing and scaling plots;
- complete the reflection cell;
- if you ran the GPU extension, keep the OpenACC compiler and GPU output too.

A practical submission format is:

1. the completed `.ipynb`;
2. an exported HTML copy so the executed outputs are visible without rerunning the notebook.

### What this lab demonstrated

```text
same scientific problem
        |
        +--> serial baseline
        |
        +--> OpenMP: shared-memory threads
        |
        +--> MPI: independent ranks + halo messages
        |
        +--> OpenACC: GPU offload + data residence
```

The key lesson is not that one model is universally "best".

The key lesson is:

> **Choose a programming model that matches where the data lives, how workers communicate, and which hardware resources the workload can use efficiently.**